# Plot Constructor Example

This notebook demonstrates how to build stacked plots from processor results.

## 1) User parameters

Set everything in one place: date, optional station codes, SIMURG email and requested plots.

In [ ]:
DATE_STR = "2025-11-12"
BASE_DIR = "files"

# Optional data-source parameters
IONOSONDE_CODE = None
COSMIC_STATIONS = ["OULU", "APTY"]
SIMURG_EMAIL = "Storm_Plotter_Jupyter_Notebook@gmail.com"

# Requested plots for PlotConstructor (order matters)
PLOTS_TO_DRAW = [
    "ROTI",
    "Aurora Observation",
    "Dst",
    "Kp",
]

# Per-plot visualization params
PLOT_SPECS = [
    {"name": "ROTI", "params": {"cmap": "viridis", "s": 10}},
    {"name": "Aurora Observation", "params": {"color": "tab:purple", "s": 60}},
    {"name": "Dst", "params": {"color": "black", "linewidth": 1.2}},
    {"name": "Kp", "params": {"bins": 9}},
]

## 2) Data loader/orchestrator (decomposed)

This helper inspects requested plot names, downloads only required sources, processes them with corresponding Processor classes and returns `processor_results`.

In [ ]:
from __future__ import annotations

import os
from datetime import datetime, timedelta
from dataclasses import dataclass
from typing import Any

import pandas as pd

from app.gfz.gfz_downloader import GfzDownloader
from app.gfz.gfz_processor import GfzProcessor
from app.kyoto.kyoto_dst_downloader import KyotoDstDownloader
from app.kyoto.kyoto_dst_processor import KyotoProcessor
from app.simurg.gim_downloader import GimDownloader
from app.simurg.gim_processor import GimProcessor
from app.simurg.simurg_client import SimurgClient
from app.simurg.simurg_downloader import RotiDownloader, AdjustedTecDownloader
from app.simurg.simurg_processor import SimurgProcessor, DataProduct


@dataclass
class ConstructorDataConfig:
    date_str: str
    base_dir: str = "files"
    simurg_email: str | None = None


class PlotConstructorDataLoader:
    def __init__(self, config: ConstructorDataConfig) -> None:
        self.config = config
        self.date_dir = os.path.join(config.base_dir, config.date_str)

    @staticmethod
    def _normalize(name: str) -> str:
        return " ".join(name.lower().replace("_", " ").split())

    def _contains(self, requested: set[str], *names: str) -> bool:
        return any(self._normalize(name) in requested for name in names)

    def _safe_download(self, fn):
        try:
            return fn()
        except Exception as exc:
            print(f"Download warning: {exc}")
            return None

    def _load_kp(self):
        out_dir = os.path.join(self.date_dir, "kp")
        os.makedirs(out_dir, exist_ok=True)
        self._safe_download(lambda: GfzDownloader(out_dir=out_dir).download(date_str=self.config.date_str, fmt="kp2"))
        return GfzProcessor(folder_path=out_dir).load(date_str=self.config.date_str)

    def _load_dst(self):
        out_dir = os.path.join(self.date_dir, "kyoto")
        os.makedirs(out_dir, exist_ok=True)
        self._safe_download(lambda: KyotoDstDownloader(out_dir=out_dir).download(self.config.date_str))
        return KyotoProcessor(folder_path=out_dir).load(self.config.date_str)

    def _simurg_client(self) -> SimurgClient | None:
        if not self.config.simurg_email:
            return None
        return SimurgClient(email=self.config.simurg_email)

    def _load_roti(self):
        client = self._simurg_client()
        if client is None:
            print("SIMURG email is missing, skip ROTI download")
            return None

        out_dir = os.path.join(self.date_dir, "simurg")
        os.makedirs(out_dir, exist_ok=True)
        self._safe_download(lambda: RotiDownloader(client=client, out_dir=out_dir).download(self.config.date_str))

        target_date = datetime.strptime(self.config.date_str, "%Y-%m-%d").date() - timedelta(days=1)
        return SimurgProcessor(folder_path=out_dir).load(target_date, product_type=DataProduct.ROTI)

    def _load_adjusted_tec(self):
        client = self._simurg_client()
        if client is None:
            print("SIMURG email is missing, skip adjusted TEC download")
            return None

        out_dir = os.path.join(self.date_dir, "simurg")
        os.makedirs(out_dir, exist_ok=True)
        self._safe_download(lambda: AdjustedTecDownloader(client=client, out_dir=out_dir).download(self.config.date_str))
        return SimurgProcessor(folder_path=out_dir).load(self.config.date_str, product_type=DataProduct.TEC_ADJUSTED)

    def _load_gim(self):
        out_dir = os.path.join(self.date_dir, "gim")
        os.makedirs(out_dir, exist_ok=True)
        self._safe_download(lambda: GimDownloader(out_dir=out_dir).download(self.config.date_str))
        return GimProcessor(folder_path=out_dir).load(self.config.date_str)

    @staticmethod
    def _aurora_stub(date_str: str) -> pd.DataFrame:
        # Minimal in-notebook demo object compatible with map plotting expectations
        return pd.DataFrame([
            {"date": date_str, "time": "02:00", "lat": 65.0, "lon": 25.0, "colors": "Green;Red"}
        ])

    def load_for_requested_plots(self, plots: list[str | dict[str, Any]]) -> dict[str, Any]:
        names = {
            self._normalize(item if isinstance(item, str) else item.get("name", ""))
            for item in plots
        }

        results: dict[str, Any] = {}

        if self._contains(names, "kp"):
            results["Kp"] = self._load_kp()
        if self._contains(names, "dst"):
            results["Dst"] = self._load_dst()
        if self._contains(names, "roti", "keogram"):
            results["ROTI"] = self._load_roti()
        if self._contains(names, "adjusted tec", "tec adjusted"):
            results["Adjusted TEC"] = self._load_adjusted_tec()
        if self._contains(names, "gim"):
            results["GIM"] = self._load_gim()
        if self._contains(names, "aurora observation", "aurora"):
            results["Aurora Observation"] = self._aurora_stub(self.config.date_str)

        return results

## 3) Build processor results for constructor

In [ ]:
from app.visualization import PlotConstructor

loader = PlotConstructorDataLoader(
    ConstructorDataConfig(
        date_str=DATE_STR,
        base_dir=BASE_DIR,
        simurg_email=SIMURG_EMAIL,
    )
)

processor_results = loader.load_for_requested_plots(PLOT_SPECS)
plotter = PlotConstructor(processor_results)
plotter.available_plots()

## 4) Plot stacked charts (same order as input list)

In [ ]:
plotter.plot(PLOT_SPECS);

## 5) Reorder plots

In [ ]:
plotter.plot([
    "Dst",
    "ROTI",
    "Kp",
]);

## 6) Error handling examples

In [ ]:
# Unknown plot name -> ValueError with available names
# plotter.plot(["UNKNOWN"])

# Missing data source -> ValueError
# PlotConstructor({"Dst": None}).plot(["Dst"])